In [ ]:
import os
import io
import PIL
import tqdm
import utils
import model
import torch
import layers
import requests
from torch import optim
from IPython import display
from torch.utils import data
from datasets import load_dataset
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader

In [ ]:
class HQ50KDataset(Dataset):
    def __init__(self, transform=None, download=False):
        self.data = self.load_data(download)
        self.transform = transform
    
    def load_data(self, download=False):
        if os.path.exists('HQ-50K'):
            print("Dataset already exists locally.")
        else:
            os.makedirs('HQ-50K')
            os.makedirs('HQ-50K/class_0')
            links = load_dataset("YangQiee/HQ-50K")['train']['text']
            for link in tqdm.notebook.tqdm(links.values(), "Downloading images"):
                try:
                    img_data = requests.get(link, timeout=5).content
                    img = PIL.Image.open(io.BytesIO(img_data)).convert('RGB')
                    img_name = os.path.join('HQ-50K/class_0', os.path.basename(link))
                    os.makedirs(os.path.dirname(img_name), exist_ok=True)
                    img.save(img_name)
                except Exception as e:
                    print(f"Failed to download {link}: {e}")
        return datasets.ImageFolder('HQ-50K', transform=self.transform)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = model.UNet(3, 128, 256, 512, MoEBlock_type=layers.DynamicMoEBlock, k=1).to(device)
m = model.DDPM(m)
opt = optim.Adam(m.parameters(), 2e-4)

batch_size = 10

class CostomTransform:
    def __init__(self, num_downsamples):
        self.num_downsamples = num_downsamples

    def __call__(self, img): # check if image size is divisible by 2**num_downsamples
        min_size = 2**self.num_downsamples
        w, h = img.size
        new_w = ((w + min_size - 1) // min_size) * min_size
        new_h = ((h + min_size - 1) // min_size) * min_size
        pad_w = new_w - w
        pad_h = new_h - h
        img = transforms.Pad((0, 0, pad_w, pad_h))(img)
        img = transforms.ToTensor()(img)
        return img


tf = CostomTransform(4)
train_set = datasets.CIFAR10('data', train=True, download=True, transform=tf)
train_dl = data.DataLoader(train_set, batch_size, shuffle=True)


In [ ]:
grid = utils.demo(m, (1024, 1024), 20, 3, filename='demo_0.png')
display.display(display.Image('demo_0.png'))
print()
for i in range(7):
    utils.train(m, train_dl, opt)
    utils.save_model(m, opt, i, f'model_{i+1}.pth')
    grid = utils.demo(m, (1024, 1024), 20, 3, filename=f'demo_{i+1}.png')
    display.display(display.Image(f'demo_{i+1}.png'))
    print()